# Neutral X追加パイロット（68入力・340回答）

[Colabで開く](https://colab.research.google.com/github/ryouy/GraduationThesis/blob/main/experiments/2026-11/1112_qwen_suppression_pilot/qwen_neutral_x_addon_4concepts.ipynb) · [事前固定した設計](https://github.com/ryouy/GraduationThesis/blob/main/docs/neutral_x_control_decision_v0_1.md)

既存v1は改変しない。**Neutral Xのみ**追加生成。4概念×（64 Cue＋Cueなし4）＝68入力、各5反復、合計340回答。既存と同じQwenのrevision、Cue、学習済みProbe、seed、生成設定、Logit Lensを使用。17ではなく5個のチェックポイントに保存。結果は専用ディレクトリに保存する。ColabのGPUとSecretsのGITHUB_RESULTS_TOKENが必要。コードを開いただけではGPU推論は実行されない。


In [ ]:
%pip -q install "transformers==4.57.6" "accelerate>=1,<2" "huggingface-hub>=0.30,<1" "scikit-learn==1.6.1" pandas numpy requests tqdm matplotlib scipy tabulate

## 0. セットアップ・原データの凍結

今回の64 CueだけをGitHubから読み込む。変更追跡版の原本を使用し、未採用Cueや生成結果に基づく再選定はしない。元のWikipediaから作った学習特徴は別のGitHubチェックポイントから**読み取り専用**で取り出す。

In [ ]:
import os, io, re, json, math, hashlib, zipfile, base64, time, unicodedata, random
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from google.colab import userdata

REPO='ryouy/GraduationThesis'
ROOT='experiments/2026-11/1112_qwen_suppression_pilot'
RUN_ID='20261124_qwen_neutral_x_addon_v1'
DEST=f'{ROOT}/results/{RUN_ID}'
CUE_SOURCE='experiments/2026-11/1112_cue_semantic_distance/final_stimuli/20261112_frozen_ai_audit_v1'
PROBE_SOURCE='experiments/2026-11/1111_qwen_probe_100concepts_v2/results/20261112_qwen100_richer_dedup_v2'
CONFIRM_SOURCE='experiments/2026-11/1112_qwen_probe_independent_confirmation/results/20261112_newfacts_16targets_v1'
MODEL_ID='Qwen/Qwen2.5-0.5B-Instruct'
MODEL_REVISION='7ae557604adf67be50417f59c2c2f167def9a775'
LAYERS=[22,23,24]
CONCEPT_Y={'boat':'bus','bus':'helicopter','bicycle':'submarine','motorcycle':'helicopter'}
CONDITIONS=['Neutral X']
BANDS=['near','medium','far','unrelated','none']
SEEDS=[101,211,307,401,503]
TEMP=0.7; TOP_P=0.9; MAX_NEW_TOKENS=128
CHUNK_INPUTS=16
WORK=Path('/content/qwen_neutral_x_addon')/RUN_ID
CACHE=WORK/'checkpoints';CACHE.mkdir(parents=True,exist_ok=True)
OUT=WORK/'results';OUT.mkdir(parents=True,exist_ok=True)
np.random.seed(42);random.seed(42);torch.manual_seed(42)
try: GITHUB_TOKEN=userdata.get('GITHUB_RESULTS_TOKEN')
except Exception: GITHUB_TOKEN=None
assert GITHUB_TOKEN, 'Please enable GITHUB_RESULTS_TOKEN in Colab Secrets (repository Contents read/write).'
G=requests.Session();G.headers.update({'Authorization':'Bearer '+GITHUB_TOKEN,'Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'})
API='https://api.github.com/repos/'+REPO+'/contents/'

def remote_meta(path):
    response=G.get(API+path,timeout=90)
    if response.status_code==404:return None
    response.raise_for_status();return response.json()

def read_remote(path):
    info=remote_meta(path)
    if info is None:raise FileNotFoundError(path)
    response=G.get(info['download_url'],timeout=240)
    response.raise_for_status();return response.content

def maybe_remote(path):
    info=remote_meta(path)
    if info is None:return None
    response=G.get(info['download_url'],timeout=240)
    response.raise_for_status();return response.content

def publish_immutable(path,data,message):
    if isinstance(data,str):data=data.encode('utf-8')
    git_sha=hashlib.sha1(b'blob '+str(len(data)).encode()+b'\0'+data).hexdigest()
    info=remote_meta(path)
    if info is not None:
        if info['sha']!=git_sha:raise RuntimeError('Different immutable artifact exists: '+path+'. Use a NEW RUN_ID.')
        print('Verified existing:',path);return
    payload={'branch':'main','message':message,'content':base64.b64encode(data).decode('ascii')}
    response=G.put(API+path,json=payload,timeout=240)
    response.raise_for_status();print('Saved:',path)

def sha256(x):return hashlib.sha256(x).hexdigest()
def dump_json(obj):return (json.dumps(obj,ensure_ascii=False,indent=2,sort_keys=True)+'\n').encode()
def save_df(df,name):df.to_csv(OUT/name,index=False)

def jsonl_bytes(rows):return ('\n'.join(json.dumps(x,ensure_ascii=False,sort_keys=True) for x in rows)+'\n').encode()
def read_jsonl(blob):return [json.loads(line) for line in blob.decode('utf-8').splitlines() if line.strip()]
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT SELECTED')

## 1. Cue原本とNeutral Xだけの入力を構築

既存64 Cueを再利用する。新条件の指示文は固定：`The word is {X}. Read the clue and briefly name the thing it brings to mind.`。既存結果は書き換えない。


In [ ]:
cue_raw=read_remote(CUE_SOURCE+'/cue_final_FROZEN_AI_AUDITED.csv')
cue=pd.read_csv(io.BytesIO(cue_raw),keep_default_na=False)
cue_manifest=json.loads(read_remote(CUE_SOURCE+'/curation_manifest.json'))
assert cue_manifest['status']=='FROZEN_FOR_PILOT_AI_REVIEWED_HUMAN_PENDING'
assert cue_manifest['final_count']==64 and not cue_manifest['human_verified']
assert len(cue)==64 and cue.cue_id.nunique()==64
assert set(cue.target_x)==set(CONCEPT_Y)
assert all(len(g)==16 for _,g in cue.groupby('target_x'))
assert cue.apply(lambda r: CONCEPT_Y[r.target_x]==r.comparator_y,axis=1).all()
assert cue.apply(lambda r: sum((cue.target_x==r.target_x)&(cue.distance_band==r.distance_band))==4,axis=1).all()
assert set(cue.distance_band)==set(BANDS)-{'none'}
assert not cue.cue_text.duplicated().any()
assert not cue.cue_text.str.contains(r'\b(?:boat|bus|bicycle|motorcycle|helicopter|submarine)\b',case=False,regex=True).any(), 'Explicit label present in cue'

# Treat all cues as distinct trial stimuli; none is added separately to avoid changing the frozen 64.
INSTRUCTIONS={
 'Base': 'Read the clue and briefly name the thing it brings to mind.',
 'Think X': 'Think about {X}. Read the clue and briefly name the thing it brings to mind.',
 'Avoid X': 'Do not think about {X}. Read the clue and briefly name the thing it brings to mind.',
 'Think Y': 'Think about {Y} instead of {X}. Read the clue and briefly name the thing it brings to mind.',
 'Neutral X': 'The word is {X}. Read the clue and briefly name the thing it brings to mind.'
}

def construct_prompt(x,y,condition,cue_text):
    inst=INSTRUCTIONS[condition].replace('{X}',x).replace('{Y}',y)
    tail='Clue: '+cue_text+'\nAnswer with one short phrase.' if cue_text else 'No clue is provided.\nAnswer with one short phrase.'
    return inst+'\n'+tail

rows=[]
for x in sorted(CONCEPT_Y):
    y=CONCEPT_Y[x]
    subset=cue[cue.target_x==x].sort_values('cue_id')
    for r in subset.to_dict('records'):
        for cond in CONDITIONS:
            rows.append({'input_id':x+'__'+r['cue_id']+'__'+cond.replace(' ','_'),
                'target_x':x,'comparator_y':y,'cue_id':r['cue_id'],'cue_text':r['cue_text'],
                'distance_band':r['distance_band'],'distance_x':float(r['distance_x']),
                'distance_y':float(r['distance_y']),'delta_x_minus_y':float(r['delta_x_minus_y']),
                'condition':cond,'is_no_cue':False,
                'user_prompt':construct_prompt(x,y,cond,r['cue_text'])})
    for cond in CONDITIONS:
        rows.append({'input_id':x+'__NO_CUE__'+cond.replace(' ','_'),
            'target_x':x,'comparator_y':y,'cue_id':'NO_CUE','cue_text':'',
            'distance_band':'none','distance_x':np.nan,'distance_y':np.nan,'delta_x_minus_y':np.nan,
            'condition':cond,'is_no_cue':True,
            'user_prompt':construct_prompt(x,y,cond,'')})
PROMPTS=pd.DataFrame(rows).sort_values(['target_x','cue_id','condition']).reset_index(drop=True)
assert len(PROMPTS)==68 and PROMPTS.input_id.is_unique
assert (PROMPTS.condition.value_counts()==68).all()
assert PROMPTS.is_no_cue.sum()==4
PROMPT_BYTES=PROMPTS.to_csv(index=False).encode()
PROMPT_HASH=sha256(PROMPT_BYTES); CUE_HASH=sha256(cue_raw)
save_df(PROMPTS,'prompts.csv')
save_df(cue,'cue_selected.csv')
print('Cue SHA256:',CUE_HASH,'Prompt SHA256:',PROMPT_HASH)
print(PROMPTS[['target_x','distance_band','condition']].groupby(['target_x','distance_band']).size().unstack())

## 2. 以前の学習Hidden Stateを復元してProbeを凍結

前回の`train`だけを復元。**評価データは分類器に入れず**、本パイロットの指示条件を使った再学習も行わない。過去の16文中8学習文、scaler + LogisticRegression(C=0.1)を同じ実装で再構成する。

In [ ]:
old_manifest=json.loads(read_remote(PROBE_SOURCE+'/manifest.json'))
old_raw=read_remote(PROBE_SOURCE+'/stimulus_design.csv')
assert old_manifest['status']=='COMPLETED_EXPLORATORY_SCREEN'
assert old_manifest['revision']==MODEL_REVISION and old_manifest['model']==MODEL_ID
assert old_manifest['layers']==LAYERS and old_manifest['position']=='assistant_start'
assert sha256(old_raw)==old_manifest['prompt_hash']
old_stim=pd.read_csv(io.BytesIO(old_raw),keep_default_na=False)
assert len(old_stim)==1104
REF=pd.read_csv(io.BytesIO(read_remote(CONFIRM_SOURCE+'/frozen_pair_confirmation.csv')))
for x,y in CONCEPT_Y.items():
    p='__'.join(sorted((x,y)))
    row=REF[REF.pair==p]
    assert len(row)==1 and bool(row.iloc[0].confirmation_pass_fdr), 'Frozen contrast did not pass independent sensitivity check: '+p
print('Frozen comparisons validated:',[(x,CONCEPT_Y[x]) for x in sorted(CONCEPT_Y)])

PREV_BATCH=64
N_OLD=len(old_stim)
N_OLD_BATCH=math.ceil(N_OLD/PREV_BATCH)
N_OLD_GROUP=math.ceil(N_OLD_BATCH/4)
OLD_FEATURES=[]
for group in tqdm(range(N_OLD_GROUP),desc='Read previous immutable training checkpoints'):
    blob=read_remote(f'{PROBE_SOURCE}/checkpoints/chunk_{group:03d}.zip')
    with zipfile.ZipFile(io.BytesIO(blob)) as archive:
        cfg=json.loads(archive.read('config.json'))
        assert cfg['prompt_hash']==old_manifest['prompt_hash'] and cfg['layers']==LAYERS
        for b in range(group*4,min((group+1)*4,N_OLD_BATCH)):
            with np.load(io.BytesIO(archive.read(f'batch_{b:03d}.npz')),allow_pickle=False) as z:
                assert str(z['prompt_hash'].item())==old_manifest['prompt_hash']
                assert int(z['start'])==b*PREV_BATCH
                assert int(z['stop'])==min(N_OLD,(b+1)*PREV_BATCH)
                v=z['features'].copy()
                assert v.shape==(min(N_OLD,(b+1)*PREV_BATCH)-b*PREV_BATCH,3,896)
                OLD_FEATURES.append(v)
OLD_FEATURES=np.concatenate(OLD_FEATURES,axis=0)
assert OLD_FEATURES.shape==(N_OLD,3,896)
OLD_INDEX={(r.concept,r.split,int(r.family)):i for i,r in enumerate(old_stim.itertuples())}
assert len(OLD_INDEX)==len(old_stim)
def old_train(name,layer_index):
    return np.stack([OLD_FEATURES[OLD_INDEX[name,'train',k],layer_index] for k in range(8)])
PROBES={};probe_records=[]
for x,y in CONCEPT_Y.items():
    for li,layer in enumerate(LAYERS):
        X=np.concatenate([old_train(x,li),old_train(y,li)],axis=0)
        labels=np.r_[np.ones(8,dtype=int),np.zeros(8,dtype=int)]
        scaler=StandardScaler().fit(X)
        clf=LogisticRegression(C=0.1,solver='liblinear',max_iter=1000,random_state=42).fit(scaler.transform(X),labels)
        PROBES[x,layer]=(scaler,clf)
        checksum=sha256(np.concatenate([scaler.mean_,scaler.scale_,clf.coef_.ravel(),clf.intercept_]).tobytes())
        refrow=REF[REF.pair=='__'.join(sorted((x,y)))].iloc[0]
        probe_records.append({'target_x':x,'comparator_y':y,'layer':layer,
          'train_examples_per_label':8,'positive_label':x,'negative_label':y,
          'probe_param_sha256':checksum,'prior_fresh_L24_auc':float(refrow.L24_auc),
          'prior_confirmation_pass_fdr':bool(refrow.confirmation_pass_fdr)})
PROBE_INFO=pd.DataFrame(probe_records)
save_df(PROBE_INFO,'probe_fit_validation.csv')
assert len(PROBES)==12
print(PROBE_INFO[['target_x','comparator_y','layer','prior_fresh_L24_auc']].to_string(index=False))
del OLD_FEATURES

## 3. 実験設定をGitHubに先に固定する

同一RUN_IDで再実行した場合はファイル内容のSHAを確認。異なるデータに変更して上書きすることはできない。

In [ ]:
CONFIG={
 'run_id':RUN_ID,'experiment_phase':'neutral_x_addon_pilot','baseline_run_id':'20261112_qwen_suppression_pilot_v1','model':MODEL_ID,'model_revision':MODEL_REVISION,
 'probe_training_source':PROBE_SOURCE,'probe_original_prompt_hash':old_manifest['prompt_hash'],
 'confirmation_source':CONFIRM_SOURCE,'stimulus_source':CUE_SOURCE,
 'cue_sha256':CUE_HASH,'prompts_sha256':PROMPT_HASH,
 'fixed_pairs':CONCEPT_Y,'conditions':CONDITIONS,'n_inputs':len(PROMPTS),
 'n_repeats':len(SEEDS),'seeds':SEEDS,'temperature':TEMP,'top_p':TOP_P,
 'max_new_tokens':MAX_NEW_TOKENS,'chunk_size_inputs':CHUNK_INPUTS,
 'primary_hidden_position':'last_input_token_in_assistant_prefix',
 'probe_layers':LAYERS,'logit_lens_layers':list(range(1,25)),
 'probe_spec':'StandardScaler + LogisticRegression(C=0.1,solver=liblinear,random_state=42)',
 'explicit_dictionary_version':'pilot-v1','ai_cue_review_only':True,
 'semantic_grader':'optional, separately versioned; not part of immutable core result'
}
publish_immutable(DEST+'/config.json',dump_json(CONFIG),'Freeze Qwen suppression pilot config')
for name in ['prompts.csv','cue_selected.csv','probe_fit_validation.csv']:
    publish_immutable(DEST+'/'+name,(OUT/name).read_bytes(),'Freeze Qwen suppression pilot '+name)

## 4. GPU実行

新条件68入力のみ。入力ごとの生成前Hidden State、固定Probe、全24層Logit Lens、5反復の最終回答を取得する。


In [ ]:
assert torch.cuda.is_available(),'Select GPU / T4 runtime in Colab.'
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=MODEL_REVISION,use_fast=True)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=MODEL_REVISION,torch_dtype=torch.float32,attn_implementation='eager').to('cuda').eval()
assert model.config.num_hidden_layers==24 and model.config.hidden_size==896
assert hasattr(model.model,'norm') and hasattr(model,'lm_head')
EOT_IDS=[i for i in [tokenizer.eos_token_id,tokenizer.convert_tokens_to_ids('<|im_end|>')] if isinstance(i,int) and i>=0]
EOT_IDS=sorted(set(EOT_IDS))

def encode_prompt(text):
    return tokenizer.apply_chat_template([{'role':'user','content':text}],tokenize=True,
       add_generation_prompt=True,return_tensors='pt').to('cuda')

def hidden_to_lens_logits(hidden,layer):
    # Qwen2 final hidden state (layer 24) has already passed final RMSNorm.
    vec=hidden if layer==24 else model.model.norm(hidden)
    return model.lm_head(vec).float()

@torch.inference_mode()
def lens_for_label(input_ids,original_forward,label):
    tokens=tokenizer.encode(label,add_special_tokens=False)
    assert 1<=len(tokens)<=8,(label,tokens)
    if len(tokens)==1:
        states=original_forward.hidden_states
    else:
        append=torch.tensor([tokens[:-1]],device='cuda',dtype=input_ids.dtype)
        extended=torch.cat((input_ids,append),dim=1)
        states=model(input_ids=extended,output_hidden_states=True,use_cache=False).hidden_states
    out=[]
    for layer in range(1,25):
        # Last n positions = last original prefix position followed by n-1 appended token positions.
        h=states[layer][0,-len(tokens):,:]
        logits=hidden_to_lens_logits(h,layer)
        logp=F.log_softmax(logits,dim=-1)
        chosen=logp[torch.arange(len(tokens),device='cuda'),torch.tensor(tokens,device='cuda')]
        out.append({'layer':layer,'sequence_logprob':float(chosen.sum().item()),
                    'mean_token_logprob':float(chosen.mean().item()),
                    'first_token_logit':float(logits[0,tokens[0]].item())})
    return tokens,out

@torch.inference_mode()
def execute_input(r):
    inp=encode_prompt(r.user_prompt)
    fwd=model(input_ids=inp,output_hidden_states=True,use_cache=False)
    # Correctness guard: layer-24 hidden is already normalized and should match the native logits.
    exact=model.lm_head(fwd.hidden_states[-1][0,-1,:]).float()
    native=fwd.logits[0,-1,:].float()
    err=float((exact-native).abs().max().item())
    assert err<1e-4, f'Final-layer Logit Lens mismatch: {err}'
    feat=np.stack([fwd.hidden_states[L][0,-1,:].float().cpu().numpy() for L in LAYERS]).astype(np.float32)
    probe=[]
    for idx,layer in enumerate(LAYERS):
        ss,clf=PROBES[r.target_x,layer]
        score=float(clf.decision_function(ss.transform(feat[idx:idx+1]))[0])
        probe.append({'input_id':r.input_id,'target_x':r.target_x,'comparator_y':r.comparator_y,
           'layer':layer,'x_vs_y_decision':score})
    lenses=[]
    for name,label in [('X',r.target_x),('Y',r.comparator_y)]:
        ids,quant=lens_for_label(inp,fwd,label)
        for ent in quant:
            lenses.append({'input_id':r.input_id,'candidate_role':name,'candidate_text':label,
               'token_ids':' '.join(map(str,ids)),'token_count':len(ids),**ent})
    del fwd
    if torch.cuda.is_available():torch.cuda.empty_cache()
    generations=[]
    for repeat,seed in enumerate(SEEDS):
        # transformers GenerationMixin does not reliably accept generator= as a kwarg.
        # Seed PyTorch immediately before sampling; document GPU nondeterminism separately.
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        sampled=model.generate(input_ids=inp,do_sample=True,temperature=TEMP,top_p=TOP_P,
            max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            eos_token_id=EOT_IDS,use_cache=True)
        output_ids=sampled[0,inp.shape[1]:].tolist()
        output_text=tokenizer.decode(output_ids,skip_special_tokens=True).strip()
        generations.append({'response_id':r.input_id+f'__repeat{repeat}',
            'input_id':r.input_id,'target_x':r.target_x,'comparator_y':r.comparator_y,
            'cue_id':r.cue_id,'distance_band':r.distance_band,'distance_x':None if pd.isna(r.distance_x) else float(r.distance_x),
            'condition':r.condition,'repeat_index':repeat,'seed':seed,'response':output_text,
            'output_token_count':len(output_ids),'truncated_at_limit':len(output_ids)>=MAX_NEW_TOKENS and (not output_ids or output_ids[-1] not in EOT_IDS),
            'empty_response':not bool(output_text)})
    return feat,probe,lenses,generations,err

# An explicit single-input GPU smoke check is integrated into the first chunk; no data are discarded.
print('Model ready. Hidden state and native logits alignment is checked for every input.')

## 5. 5チャンクで中断再開

各16入力、最後は4入力。保存済み成果物はSHAで照合して再利用し、異なるデータでは上書きできない。


In [ ]:
N=len(PROMPTS);N_CHUNKS=math.ceil(N/CHUNK_INPUTS)
def remote_chunk(g):return f'{DEST}/checkpoints/chunk_{g:03d}.zip'
def local_chunk(g):return CACHE/f'chunk_{g:03d}.zip'

def verify_chunk(g,blob):
    begin=g*CHUNK_INPUTS;end=min(N,(g+1)*CHUNK_INPUTS)
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        cfg=json.loads(z.read('config.json'))
        assert cfg==CONFIG,'Changed config: use NEW RUN_ID'
        metadata=json.loads(z.read('meta.json'))
        assert metadata['prompt_hash']==PROMPT_HASH
        assert metadata['begin']==begin and metadata['end']==end
        with np.load(io.BytesIO(z.read('hidden_features.npz')),allow_pickle=False) as f:
            data=f['features']
            assert data.shape==(end-begin,len(LAYERS),896) and np.isfinite(data).all()
        generations=read_jsonl(z.read('generations.jsonl'))
        probes=read_jsonl(z.read('probe_scores.jsonl'))
        lenses=read_jsonl(z.read('logit_lens.jsonl'))
        assert len(generations)==(end-begin)*len(SEEDS)
        assert len(probes)==(end-begin)*len(LAYERS)
        assert len(lenses)==(end-begin)*24*2
        assert {a['input_id'] for a in generations}==set(PROMPTS.input_id.iloc[begin:end])
        assert len({a['response_id'] for a in generations})==len(generations)
    return True

def recover_chunk(g):
    remote=maybe_remote(remote_chunk(g))
    if remote is not None:
        verify_chunk(g,remote)
        local_chunk(g).write_bytes(remote)
        print('Recovered',g,'from GitHub')
        return True
    if local_chunk(g).exists():
        verify_chunk(g,local_chunk(g).read_bytes())
        return True
    return False

def execute_chunk(g):
    begin=g*CHUNK_INPUTS;end=min(N,(g+1)*CHUNK_INPUTS)
    features=[];probes=[];lenses=[];generations=[];errors=[]
    for r in tqdm(PROMPTS.iloc[begin:end].itertuples(index=False),total=end-begin,desc=f'Chunk {g:03d}'):
        f,p,l,gen,err=execute_input(r)
        features.append(f);probes.extend(p);lenses.extend(l);generations.extend(gen);errors.append(err)
    assert len(features)==end-begin
    f=io.BytesIO()
    np.savez_compressed(f,features=np.stack(features),prompt_hash=np.array(PROMPT_HASH))
    metadata={'begin':begin,'end':end,'prompt_hash':PROMPT_HASH,'final_layer_max_logit_error':max(errors)}
    buf=io.BytesIO()
    with zipfile.ZipFile(buf,'w',compression=zipfile.ZIP_DEFLATED) as z:
        z.writestr('config.json',dump_json(CONFIG))
        z.writestr('meta.json',dump_json(metadata))
        z.writestr('hidden_features.npz',f.getvalue())
        z.writestr('probe_scores.jsonl',jsonl_bytes(probes))
        z.writestr('logit_lens.jsonl',jsonl_bytes(lenses))
        z.writestr('generations.jsonl',jsonl_bytes(generations))
    raw=buf.getvalue();verify_chunk(g,raw)
    local_chunk(g).write_bytes(raw)
    # Local presence does NOT mean remote checkpoint exists. Publish after a new chunk.
    # If a remote checkpoint appeared during execution, validate and prefer it without overwriting.
    remote=maybe_remote(remote_chunk(g))
    if remote is None:
        publish_immutable(remote_chunk(g),raw,f'Qwen suppression pilot checkpoint {g:03d}')
    else:
        verify_chunk(g,remote)
        local_chunk(g).write_bytes(remote)
    return True

for g in range(N_CHUNKS):recover_chunk(g)
for g in range(N_CHUNKS):
    if local_chunk(g).exists():continue
    execute_chunk(g)
for g in range(N_CHUNKS):assert recover_chunk(g)
print('All',N_CHUNKS,'chunks complete and GitHub backed.')

## 6. 340回答・204 Probe・3,264 Logit Lens行の検証と辞書判定


In [ ]:
features=[];generations=[];probes=[];lenses=[];lens_errors=[]
for g in range(N_CHUNKS):
    verify_chunk(g,local_chunk(g).read_bytes())
    with zipfile.ZipFile(local_chunk(g),'r') as z:
        with np.load(io.BytesIO(z.read('hidden_features.npz')),allow_pickle=False) as f:
            features.append(f['features'].copy())
        generations+=read_jsonl(z.read('generations.jsonl'))
        probes+=read_jsonl(z.read('probe_scores.jsonl'))
        lenses+=read_jsonl(z.read('logit_lens.jsonl'))
        lens_errors.append(json.loads(z.read('meta.json'))['final_layer_max_logit_error'])
FEATURES=np.concatenate(features)
assert FEATURES.shape==(68,3,896)
GEN=pd.DataFrame(generations); PROBE=pd.DataFrame(probes); LENS=pd.DataFrame(lenses)
assert len(GEN)==340 and len(PROBE)==68*3 and len(LENS)==68*24*2
assert GEN.response_id.is_unique
assert PROBE[['input_id','layer']].drop_duplicates().shape[0]==len(PROBE)
assert LENS[['input_id','layer','candidate_role']].drop_duplicates().shape[0]==len(LENS)
print('Max native vs reconstructed final-layer logits mismatch:',max(lens_errors))

# High-precision dictionary. Broader paraphrase/negation handling is deliberately deferred to API grading.
ALIASES={
 'boat':['boat','boats','watercraft','dinghy','dinghies','rowboat','rowboats','motorboat','motorboats'],
 'bus':['bus','buses','coach','coaches','shuttle bus','shuttle buses'],
 'bicycle':['bicycle','bicycles','bike','bikes','pushbike','pushbikes'],
 'motorcycle':['motorcycle','motorcycles','motorbike','motorbikes','motorized two wheeler','motorized two wheelers']
}
PATTERNS={x:re.compile(r'(?<![A-Za-z])(?:'+'|'.join(re.escape(v) for v in sorted(aliases,key=len,reverse=True))+r')(?![A-Za-z])',re.IGNORECASE) for x,aliases in ALIASES.items()}
def explicit_mention(x,text):
    normalized=unicodedata.normalize('NFKC',str(text))
    return bool(PATTERNS[x].search(normalized))
GEN['explicit_mention']=[explicit_mention(row.target_x,row.response) for row in GEN.itertuples()]
GEN['format_issue']=GEN.empty_response | GEN.truncated_at_limit | (GEN.output_token_count>64)
GEN['semantic_label']='not_run'
save_df(GEN,'generations.csv');save_df(PROBE,'probe_scores.csv');save_df(LENS,'logit_lens.csv')
summary=GEN.groupby(['target_x','distance_band','condition'],dropna=False).agg(
 n=('explicit_mention','size'),explicit_rate=('explicit_mention','mean'),
 format_issue_rate=('format_issue','mean'),mean_tokens=('output_token_count','mean')).reset_index()
save_df(summary,'explicit_mention_by_condition_band.csv')
print(summary.to_string(index=False))

## 7. 旧パイロットとの対応比較

旧結果を読み取り専用で取得し、同じX・Cue・seedごとの言及率を比較する。Cueなし・無関係は独立の対照として扱う。Probeは同じX/Yペア・層で比較する。


In [ ]:
BASE_DEST=f"{ROOT}/results/20261112_qwen_suppression_pilot_v1"
OLD_GEN=pd.read_csv(io.BytesIO(read_remote(BASE_DEST+'/generations.csv')),keep_default_na=False)
OLD_PROBE=pd.read_csv(io.BytesIO(read_remote(BASE_DEST+'/probe_scores.csv')),keep_default_na=False)
OLD_LENS=pd.read_csv(io.BytesIO(read_remote(BASE_DEST+'/logit_lens.csv')),keep_default_na=False)
assert len(OLD_GEN)==1360 and len(OLD_PROBE)==816 and len(OLD_LENS)==13056
assert set(OLD_GEN.condition)=={'Base','Think X','Avoid X','Think Y'}
assert set(GEN.condition)=={'Neutral X'}
ALL_GEN=pd.concat([OLD_GEN,GEN],ignore_index=True)
assert len(ALL_GEN)==1700
CUE_RATES=ALL_GEN.groupby(['target_x','cue_id','distance_band','condition'],as_index=False).agg(
 explicit_rate=('explicit_mention','mean'),n_repeats=('explicit_mention','size'))
assert (CUE_RATES.n_repeats==5).all()
WIDE=CUE_RATES.pivot(index=['target_x','cue_id','distance_band'],columns='condition',values='explicit_rate').reset_index()
assert len(WIDE)==68 and WIDE[['Neutral X','Avoid X','Base','Think X','Think Y']].notna().all().all()
WIDE['avoid_minus_neutral']=WIDE['Avoid X']-WIDE['Neutral X']
WIDE['neutral_minus_base']=WIDE['Neutral X']-WIDE['Base']
WIDE['avoid_minus_thinkx']=WIDE['Avoid X']-WIDE['Think X']
WIDE=WIDE.merge(PROMPTS[['target_x','cue_id','distance_band','distance_x']].drop_duplicates(),
 on=['target_x','cue_id','distance_band'],validate='one_to_one')
save_df(WIDE,'paired_cue_explicit_effects.csv')
ALL_PROBE=pd.concat([OLD_PROBE,PROBE],ignore_index=True)
P_SCORE=ALL_PROBE.merge(pd.concat([pd.read_csv(io.BytesIO(read_remote(BASE_DEST+'/prompts.csv'))),PROMPTS],ignore_index=True)[['input_id','cue_id','condition','distance_band']],
 on='input_id',how='left',validate='many_to_one')
assert P_SCORE.cue_id.notna().all()
PW=P_SCORE.pivot(index=['target_x','cue_id','distance_band','layer'],columns='condition',values='x_vs_y_decision').reset_index()
assert len(PW)==204
PW['avoid_minus_neutral']=PW['Avoid X']-PW['Neutral X']
PW['neutral_minus_base']=PW['Neutral X']-PW['Base']
PW['avoid_minus_thinkx']=PW['Avoid X']-PW['Think X']
save_df(PW,'paired_cue_probe_effects.csv')
ALL_LENS=pd.concat([OLD_LENS,LENS],ignore_index=True)
LP=ALL_LENS.pivot(index=['input_id','layer'],columns='candidate_role',values='sequence_logprob').reset_index()
LP['margin_x_y']=LP['X']-LP['Y']
prom_all=pd.concat([pd.read_csv(io.BytesIO(read_remote(BASE_DEST+'/prompts.csv'))),PROMPTS],ignore_index=True)
LP=LP.merge(prom_all[['input_id','target_x','cue_id','distance_band','condition']],on='input_id',validate='many_to_one')
LW=LP.pivot(index=['target_x','cue_id','distance_band','layer'],columns='condition',values='margin_x_y').reset_index()
assert len(LW)==68*24
LW['avoid_minus_neutral']=LW['Avoid X']-LW['Neutral X']
LW['neutral_minus_base']=LW['Neutral X']-LW['Base']
save_df(LW,'paired_cue_lens_effects.csv')
RELATED=WIDE[WIDE.distance_band.isin(['near','medium','far'])]
print('Cue-level Explicit Mention differences:')
print(RELATED.groupby(['target_x','distance_band'])[['avoid_minus_neutral','neutral_minus_base','avoid_minus_thinkx']].mean().round(3).to_string())
print('Probe L24 differences:')
print(PW[(PW.layer==24)&PW.distance_band.isin(['near','medium','far'])].groupby('target_x')[['avoid_minus_neutral','neutral_minus_base']].mean().round(3).to_string())
fig,ax=plt.subplots(figsize=(8,4))
for condition in ['Base','Neutral X','Avoid X','Think X','Think Y']:
 arr=[CUE_RATES[(CUE_RATES.distance_band==b)&(CUE_RATES.condition==condition)].explicit_rate.mean() for b in ['near','medium','far']]
 ax.plot(['near','medium','far'],arr,marker='o',label=condition)
ax.set(xlabel='Cue distance band',ylabel='Explicit X mention rate',ylim=(-.03,1.03),title='Neutral X additional control')
ax.legend();fig.tight_layout();fig.savefig(OUT/'neutral_x_comparison.png',dpi=160,bbox_inches='tight');plt.show()


## 8. 別ディレクトリに結果を保存

既存結果は変更しない。追加条件340件、内部指標、対応差、実行環境、プロンプトハッシュを保存する。意味的API判定は今回行わない（別工程）。


In [ ]:
MANIFEST={**CONFIG,'status':'COMPLETED_NEUTRAL_X_CORE_PILOT','created_utc':datetime.now(timezone.utc).isoformat(),
'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,'checkpoint_count':N_CHUNKS,
'n_outputs':len(GEN),'n_probe_scores':len(PROBE),'n_lens_rows':len(LENS),
'max_final_layer_logit_error':max(lens_errors),'reference_baseline':BASE_DEST,
'human_cue_review':'PENDING','human_output_audit':'PENDING',
'limitations':['X name is matched but instruction wording and token position are not perfectly matched',
'Only four vehicle concepts; Cue specificity may differ by distance',
'Probe scores are X/Y relative classifiers, not absolute thought measurement',
'Five repeats for one Cue are not five independent stimuli',
'Comparisons are exploratory because added condition was designed after viewing v1 pilot']}
prior=maybe_remote(DEST+'/manifest.json')
if prior is not None:
 old_manifest=json.loads(prior)
 assert all(old_manifest[k]==MANIFEST[k] for k in CONFIG)
 MANIFEST=old_manifest
(OUT/'manifest.json').write_bytes(dump_json(MANIFEST))
r=RELATED.groupby(['target_x','distance_band'])[['avoid_minus_neutral','neutral_minus_base','avoid_minus_thinkx']].mean().round(4).to_markdown()
report=f"""# Neutral X追加パイロット：結果

- 状態：COMPLETED_NEUTRAL_X_CORE_PILOT
- 実行：{len(PROMPTS)}入力、{len(GEN)}生成、{len(PROBE)} Probe、{len(LENS)} Logit Lens
- Cue SHA256: `{CUE_HASH}`
- Prompts SHA256: `{PROMPT_HASH}`
- native logits最大誤差: {max(lens_errors):.8g}
- 比較先：{BASE_DEST}

## Cueごとの平均差（明示的言及）

{r}

## 解釈上の注意

これは既存v1パイロットを見た後に設計した追加の探索的対照である。
Neutral XはX名の提示を揃える一方、文法や入力位置を厳密には揃えない。
よって純粋な抑制効果の因果的証明とはしない。
結果方向によらず追加プロンプトの再選定を行わない。
人手による出力監査、Cue監査は別作業として未完了。
"""
(OUT/'analysis_summary.md').write_text(report,encoding='utf-8')
files=['manifest.json','analysis_summary.md','config.json','prompts.csv','cue_selected.csv','probe_fit_validation.csv',
'generations.csv','probe_scores.csv','logit_lens.csv','explicit_mention_by_condition_band.csv',
'paired_cue_explicit_effects.csv','paired_cue_probe_effects.csv','paired_cue_lens_effects.csv','neutral_x_comparison.png']
for name in files:
 local=(OUT/name).read_bytes();remote=maybe_remote(f'{DEST}/{name}')
 if remote is not None:
  if remote!=local:
   if name in ('analysis_summary.md','manifest.json'):
    (OUT/name).write_bytes(remote);print('Preserved immutable:',name);continue
   raise RuntimeError('Immutable output mismatch: '+name)
  print('Verified existing:',name);continue
 publish_immutable(f'{DEST}/{name}',local,'Publish Neutral X addon '+name)
print('Result:',f'https://github.com/{REPO}/tree/main/{DEST}')


## 9. 完了確認

`manifest.json`のstatus、340生成、204 Probe、3,264 Lens、5チャンク、対応集計が保存されたことを確認する。Colabを実行していない場合は未完了。
